# SAR Oil Spill — Preprocessing Notebook

Run this notebook **once** to preprocess the full dataset and save patches to Drive.
After this, the training notebook will use the saved patches directly — no need to touch the raw data again.

## Steps
1. Mount Drive and locate the dataset
2. Clone your GitHub repo
3. Install dependencies
4. Run preprocessing (generates .npy patches)
5. Save patches to Drive

**No GPU needed for this notebook** — keep runtime as CPU to save GPU quota.

---
## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

# ── Check free disk space ───────────────────────────────────────────────────
def gb(x): return x / (1024 ** 3)
total, used, free = shutil.disk_usage('/content')
print(f'Colab local disk — total: {gb(total):.1f} GB | used: {gb(used):.1f} GB | free: {gb(free):.1f} GB')

---
## Cell 2 — Locate Dataset from Shared Drive Link

Your dataset is shared at:
https://drive.google.com/drive/folders/1vKhn6wthK5ITHp3kCPBsGFvPufRPwrNW

To access it, you need to **add it to your Drive** first:
1. Open the link above
2. Click the folder name at the top
3. Click **Add shortcut to Drive** → My Drive
4. Then run this cell — it will find the folder automatically

In [ ]:
import os

# The folder ID from your shared link
FOLDER_ID = '1vKhn6wthK5ITHp3kCPBsGFvPufRPwrNW'

# After adding shortcut to Drive, the folder is accessible via mounted path.
# We search for it by folder ID using the Drive API.
# Alternatively, if you know the folder name, set DATASET_PATH directly:
#   DATASET_PATH = '/content/drive/MyDrive/YOUR_FOLDER_NAME'

# Try to find the folder by scanning MyDrive
DATASET_PATH = None
search_root = '/content/drive/MyDrive'

for item in os.listdir(search_root):
    candidate = os.path.join(search_root, item)
    # Check if this folder contains images/ and masks/ subfolders
    if os.path.isdir(candidate):
        has_images = os.path.isdir(os.path.join(candidate, 'images'))
        has_masks  = os.path.isdir(os.path.join(candidate, 'masks'))
        if has_images and has_masks:
            DATASET_PATH = candidate
            break

if DATASET_PATH:
    images = [f for f in os.listdir(f'{DATASET_PATH}/images') if f.endswith('.tif')]
    masks  = [f for f in os.listdir(f'{DATASET_PATH}/masks')  if f.endswith('.tif')]
    print(f'Dataset found at : {DATASET_PATH}')
    print(f'Images           : {len(images)}')
    print(f'Masks            : {len(masks)}')
else:
    print('ERROR: Dataset folder not found in MyDrive.')
    print('Please either:')
    print('  1. Add the shared folder as a shortcut to your Drive (recommended), OR')
    print('  2. Set DATASET_PATH manually below:')
    print("     DATASET_PATH = '/content/drive/MyDrive/your_folder_name'")

---
## Cell 3 — Clone GitHub Repository

In [ ]:
import os

# ── Replace with your actual GitHub repo URL ────────────────────────────────
GITHUB_URL = 'https://github.com/TigranBoyakhchyan/GeoSpill-AI'
REPO_NAME  = 'GeoSpill-AI'

if os.path.exists(f'/content/{REPO_NAME}'):
    print('Repo already exists — pulling latest changes...')
    %cd /content/{REPO_NAME}
    !git pull origin main
else:
    print('Cloning repository...')
    !git clone {GITHUB_URL}
    %cd /content/{REPO_NAME}

print(f'Working directory: {os.getcwd()}')

---
## Cell 4 — Install Dependencies

In [ ]:
!pip install -q rasterio

import rasterio
print(f'rasterio: {rasterio.__version__}')
print('Dependencies ready')

---
## Cell 5 — Run Preprocessing

This reads the raw `.tif` files directly from Drive (no copying needed — we only read each file once during preprocessing).

Settings:
- `patch_size = 256` — standard patch size
- `stride = 128` — 50% overlap, gives more training patches than local run

Expected time: **20-40 minutes** for 1200 images.

In [ ]:
import sys
sys.path.insert(0, '/content/oil_spill_detection')

import src.preprocess as pre

# Point to the dataset on Drive — we read .tif files once, so Drive speed is fine
pre.IMAGES_DIR = f'{DATASET_PATH}/images'
pre.MASKS_DIR  = f'{DATASET_PATH}/masks'

# Patches are saved locally on Colab disk (fast write)
pre.OUTPUT_DIR = 'data/patches'
pre.STATS_FILE = 'data/train_stats.json'

# 50% overlap — more patches than local training (stride=256)
pre.PATCH_SIZE = 256
pre.STRIDE     = 128

pre.run_preprocessing()
print('\nPreprocessing complete!')

---
## Cell 6 — Verify Patches

In [ ]:
import os, json

print('Patch counts:')
total_patches = 0
for split in ['train', 'val', 'test']:
    split_dir = f'data/patches/{split}'
    n = len([f for f in os.listdir(split_dir) if f.endswith('_img.npy')])
    total_patches += n
    print(f'  {split:<6}: {n} patches')
print(f'  Total  : {total_patches} patches')

# Check patch folder size
result = !du -sh data/patches
print(f'\nPatches folder size: {result[0].split()[0]}')

# Print normalization stats
with open('data/train_stats.json') as f:
    stats = json.load(f)
print(f'\nNormalization stats:')
print(f'  mean (VV, VH): {stats["mean"]}')
print(f'  std  (VV, VH): {stats["std"]}')

---
## Cell 7 — Save Patches to Google Drive

**Run this before closing the session.**
Saves preprocessed patches to Drive so future training sessions skip preprocessing entirely.

This copies a few GB (patches) — takes 5-15 minutes.

In [ ]:
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/oil_spill_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save patches
patches_dest = f'{SAVE_DIR}/patches'
if os.path.exists(patches_dest):
    print('Removing old patches from Drive...')
    shutil.rmtree(patches_dest)

print('Copying patches to Drive (this may take a few minutes)...')
shutil.copytree('data/patches', patches_dest)
print(f'Patches saved to: {patches_dest}')

# Save stats
shutil.copy('data/train_stats.json', f'{SAVE_DIR}/train_stats.json')
print(f'Stats saved to  : {SAVE_DIR}/train_stats.json')

# Confirm
result = !du -sh {patches_dest}
print(f'\nTotal saved to Drive: {result[0].split()[0]}')
print('\nDone! You can now close this session and run the training notebook.')